# EuRoC All-Dataset Spectrograms

Run the same basis fit over every merged EuRoC CSV and display the average spectra plus coefficient spectrogram for each dataset. This notebook is deliberately noninteractive; edit the constants in the setup cell to change basis, `N`, interval length, or signal group. Supported basis values are `chebyshev`, `chebyshev2`, and `fourier`.

The initial hardcoded signal is `gyro`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

from IPython.display import display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "python" / "imuFactors").exists():
    REPO_ROOT = Path("/Users/dellaert/git/imuFactors")

PYTHON_DIR = REPO_ROOT / "python"
if str(PYTHON_DIR) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIR))

import imuFactors.euroc as euroc
import imuFactors.spectral as spectral
import imuFactors.spectrogram as spectrogram

DATA_DIR = REPO_ROOT / "data" / "euroc"
DATA_FILES = euroc.discover_euroc_files(DATA_DIR)
if not DATA_FILES:
    raise FileNotFoundError(f"No EuRoC CSV files found in {DATA_DIR}")

SIGNAL_GROUP = "gyro"
BASIS = "chebyshev"
COEFFICIENT_COUNT = 14
WINDOW_SECONDS = 1.0
PART_COUNT = 4

print(f"Found {len(DATA_FILES)} EuRoC CSV files in {DATA_DIR}")
print(
    f"signal={SIGNAL_GROUP}, basis={BASIS}, "
    f"N={COEFFICIENT_COUNT}, interval={WINDOW_SECONDS:g}s"
)

In [ ]:
results = {}
for path in DATA_FILES:
    dataset = path.stem.removeprefix("euroc_")
    print(f"fitting {dataset}...")
    results[dataset] = spectrogram.fit_spectral_windows(
        path,
        coefficient_count=COEFFICIENT_COUNT,
        basis=BASIS,
        signal_group=SIGNAL_GROUP,
        window_seconds=WINDOW_SECONDS,
    )

print(f"fit {len(results)} datasets")

## Average Spectra and Coefficient Spectrograms

Each dataset is shown with a summary table, its average spectra, and its `m x N` coefficient spectrogram. The spectrogram time axis is reversed so time 0 appears at the top.

In [ ]:
for dataset, result in results.items():
    display(spectrogram.summary_table(result))

    average_figure = spectrogram.plot_average_spectra(
        result,
        part_count=PART_COUNT,
    )
    average_figure.update_layout(
        title=(
            f"{dataset}: {SIGNAL_GROUP} average spectra "
            f"({result.basis_name}, N={result.coefficient_count}, "
            f"{result.window_seconds:g}s intervals)"
        )
    )
    average_figure.show()

    spectrogram_figure = spectrogram.plot_coefficient_spectrogram(result)
    spectrogram_figure.update_layout(
        title=(
            f"{dataset}: {SIGNAL_GROUP} coefficient spectrogram "
            f"({result.basis_name}, N={result.coefficient_count}, "
            f"{result.window_seconds:g}s intervals)"
        )
    )
    spectrogram_figure.show()

## All-Dataset Whole-File Average Comparison

This final heatmap compares the entire-file average coefficient spectrogram for every dataset. Each row is one dataset; each column is one spectral coefficient.

In [ ]:
comparison_figure = spectrogram.plot_dataset_average_spectrograms(results)
comparison_figure.update_layout(
    title=(
        f"All datasets: {SIGNAL_GROUP} whole-file average coefficient spectrograms "
        f"({BASIS}, N={COEFFICIENT_COUNT}, {WINDOW_SECONDS:g}s intervals)"
    )
)
comparison_figure.show()